In [ ]:
"""
Лабораторная работа 1. Сетевое планирование.
 Автоматический расчёт.
"""

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import numpy as np
import os

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False

OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# 1. ИСХОДНЫЕ ДАННЫЕ
# ============================================================

raboty = [
    ("1,2", 10, 4, 0),
    ("1,4", 1, 2, 21),
    ("1,5", 5, 2, 15),
    ("2,3", 9, 3, 0),
    ("2,5", 8, 2, 2),
    ("2,6", 10, 5, 7),
    ("3,4", 3, 1, 0),
    ("3,6", 4, 2, 4),
    ("4,6", 5, 3, 0),
    ("4,7", 4, 5, 9),
    ("5,6", 7, 2, 2),
    ("5,7", 3, 2, 14),
    ("6,7", 8, 2, 0),
]

rannee_nachalo = {
    "1,2": 0, "1,4": 0, "1,5": 0,
    "2,3": 10, "2,5": 10, "2,6": 10,
    "3,4": 19, "3,6": 19,
    "4,6": 22, "4,7": 22,
    "5,6": 18, "5,7": 18,
    "6,7": 27,
}

df = pd.DataFrame(raboty, columns=["Работа", "Длит", "Ресурсы", "Резерв"])
df["Начало"] = df["Работа"].map(rannee_nachalo)
df["Конец"] = df["Начало"] + df["Длит"]
df["Ресурсо-дни"] = df["Длит"] * df["Ресурсы"]

total_res_days = df["Ресурсо-дни"].sum()
T_kr = int(df["Конец"].max())

print("=" * 70)
print("ТАБЛИЦА РАБОТ")
print("=" * 70)
print(df.to_string(index=False))
print()
print(f"Всего ресурсо-дней: {total_res_days}")
print(f"Критический путь: {T_kr} дней")
print(f"Среднее: {total_res_days} / {T_kr} = {total_res_days/T_kr:.2f}")
print()

# ============================================================
# 2. ТАБЛИЦА РЕСУРСОВ ПО ДНЯМ
# ============================================================

def build_resource_table(df, max_day):
    days = list(range(1, max_day + 1))
    table = {}
    for _, row in df.iterrows():
        row_data = []
        for d in days:
            if row["Начало"] < d <= row["Конец"]:
                row_data.append(row["Ресурсы"])
            else:
                row_data.append(0)
        table[row["Работа"]] = row_data
    table["СУММА"] = [sum(table[r][i] for r in table if r != "СУММА")
                     for i in range(len(days))]
    return pd.DataFrame(table, index=days).T

res_table = build_resource_table(df, T_kr)
print("=" * 70)
print("РАСПРЕДЕЛЕНИЕ РЕСУРСОВ (без сдвига)")
print("=" * 70)
print(res_table.to_string())
print()

# ============================================================
# 3. ДИАГРАММА ГАНТА
# ============================================================

def build_gantt(df, filename):
    fig, ax = plt.subplots(figsize=(14, 8))
    for i, row in df.iterrows():
        if row["Резерв"] > 0:
            ax.barh(i, row["Резерв"], left=row["Конец"],
                    color="white", edgecolor="black",
                    linewidth=1.2, linestyle="--", height=1.0)
        ax.barh(i, row["Длит"], left=row["Начало"],
                color="lightgray", edgecolor="black",
                linewidth=1.2, height=1.0, hatch="///")
    ax.set_yticks(range(len(df)))
    ax.set_yticklabels(df["Работа"])
    ax.set_xlabel("Дни", fontsize=12)
    ax.set_ylabel("Работы", fontsize=12)
    ax.set_title("Диаграмма Ганта", fontsize=14)
    ax.set_xlim(0, T_kr + 2)
    ax.set_xticks(range(0, T_kr + 2, 2))
    ax.grid(axis="x", linestyle=":", alpha=0.4)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    legend_elements = [
        mpatches.Patch(facecolor="lightgray", edgecolor="black",
                       hatch="///", label="Работа"),
        mpatches.Patch(facecolor="white", edgecolor="black",
                       linestyle="--", label="Резерв"),
    ]
    ax.legend(handles=legend_elements, loc="lower right")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, filename), dpi=200)
    plt.close()
    print(f"  ✅ {filename}")

build_gantt(df, "gantt.png")

# ============================================================
# 4. ГИСТОГРАММА
# ============================================================

def build_resource_hist(res_table, filename, limit=None, title_suffix=""):
    days = res_table.columns.tolist()
    values = res_table.loc["СУММА"].tolist()
    fig, ax = plt.subplots(figsize=(14, 6))
    colors = ["seagreen" if (limit is None or v <= limit) else "tomato"
              for v in values]
    ax.bar(days, values, color=colors, edgecolor="black", width=0.9)
    if limit is not None:
        ax.axhline(limit, color="red", linestyle="--", linewidth=2,
                   label=f"Лимит = {limit}")
    sred = sum(values) / len(values)
    ax.axhline(sred, color="blue", linestyle=":", linewidth=1.5,
               label=f"Среднее = {sred:.2f}")
    ax.set_xlabel("Дни", fontsize=12)
    ax.set_ylabel("Ресурсы, чел.", fontsize=12)
    ax.set_title(f"Распределение ресурсов {title_suffix}", fontsize=14)
    ax.set_xticks(days)
    ax.set_ylim(0, max(values) + 2)
    ax.grid(axis="y", linestyle=":", alpha=0.4)
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, filename), dpi=200)
    plt.close()
    print(f"  ✅ {filename}")

build_resource_hist(res_table, "resursy_bez_sdviga.png",
                    title_suffix="(без сдвига)")

# ============================================================
# 5. АВТОМАТИЧЕСКИЙ СДВИГ
# ============================================================

def shift_works(df, limit, max_day):
    """
    Сдвигает некритические работы, чтобы пик <= limit.
    Возвращает (df_shifted, success).
    """
    df = df.copy()
    # Сортируем по резерву (сначала с большим — их можно сдвинуть дальше)
    order = df[df["Резерв"] > 0].sort_values("Резерв", ascending=False).index.tolist()

    # Итеративно: пока пик > limit, пробуем сдвинуть
    max_iter = 100
    for _ in range(max_iter):
        table = build_resource_table(df, max_day)
        peak = table.loc["СУММА"].max()
        if peak <= limit:
            return df, True

        # Ищем, какую работу сдвинуть
        moved = False
        for idx in order:
            current_start = df.loc[idx, "Начало"]
            reserve = df.loc[idx, "Резерв"]
            for shift in range(1, reserve + 1):
                new_start = current_start + shift
                new_end = new_start + df.loc[idx, "Длит"]
                if new_end > max_day:
                    break
                test_df = df.copy()
                test_df.loc[idx, "Начало"] = new_start
                test_df.loc[idx, "Конец"] = new_end
                test_table = build_resource_table(test_df, max_day)
                test_peak = test_table.loc["СУММА"].max()
                if test_peak < peak:
                    df = test_df
                    moved = True
                    break
            if moved:
                break
        if not moved:
            return df, False

    return df, False

# ============================================================
# 6. ПРОБУЕМ РАЗНЫЕ ЛИМИТЫ
# ============================================================

limits = [12, 11, 10, 9, 8, 7, 6]
results = {}

print()
print("=" * 70)
print("СДВИГ РАБОТ ДЛЯ РАЗНЫХ ЛИМИТОВ")
print("=" * 70)

for lim in limits:
    df_shifted, success = shift_works(df, lim, T_kr)
    table = build_resource_table(df_shifted, T_kr)
    peak = table.loc["СУММА"].max()

    results[lim] = {
        "df": df_shifted,
        "table": table,
        "peak": peak,
        "success": success,
    }

    status = "✅" if peak <= lim else "❌"
    print(f"Лимит = {lim}: пик = {peak} {status}")

    # Сохраняем таблицу в Excel
    table.to_excel(os.path.join(OUTPUT_DIR, f"resursy_{lim}.xlsx"))
    df_shifted.to_excel(os.path.join(OUTPUT_DIR, f"raboty_{lim}.xlsx"), index=False)

    # Гистограмма
    build_resource_hist(
        table,
        f"resursy_{lim}.png",
        limit=lim,
        title_suffix=f"(лимит = {lim})",
    )

# ============================================================
# 7. ГРАФИК КРЕДИТА
# ============================================================

fig, ax = plt.subplots(figsize=(14, 8))
x = np.linspace(0, 800, 500)
kredit = (10/9) * x + 2000
pribyl_05 = 10 * x - 4655
pribyl_05[pribyl_05 < 0] = np.nan
pribyl_2 = 40 * x - 6205
pribyl_2[pribyl_2 < 0] = np.nan

ax.plot(x, kredit, "b-", linewidth=2.5, label="Кредит")
ax.plot(x, pribyl_05, "r-", linewidth=2.5, label="Прибыль (0,5 д.е./т)")
ax.plot(x, pribyl_2, "g--", linewidth=2.5, label="Прибыль (2 д.е./т)")
ax.plot(748.7, 2832, "ko", markersize=10)
ax.annotate("(748,7; 2832)", xy=(748.7, 2832),
            xytext=(550, 2600),
            arrowprops=dict(arrowstyle="->", color="black", lw=1.5),
            fontsize=12)
ax.set_xlabel("Дни", fontsize=13)
ax.set_ylabel("Стоимость, д.е.", fontsize=13)
ax.set_title("Зависимость кредита и прибыли", fontsize=14)
ax.set_xlim(0, 800)
ax.set_ylim(-2000, 3500)
ax.grid(True, linestyle=":", alpha=0.5)
ax.legend(loc="upper left", fontsize=12)
ax.axhline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "credit.png"), dpi=200)
plt.close()
print("  ✅ credit.png")

# ============================================================
# 8. ИТОГОВЫЙ EXCEL
# ============================================================

with pd.ExcelWriter(os.path.join(OUTPUT_DIR, "otchet.xlsx")) as writer:
    df.to_excel(writer, sheet_name="Работы", index=False)
    res_table.to_excel(writer, sheet_name="Ресурсы_без_сдвига")
    for lim in limits:
        results[lim]["table"].to_excel(writer, sheet_name=f"Ресурсы_{lim}")
        results[lim]["df"].to_excel(writer, sheet_name=f"Работы_{lim}", index=False)

print()
print("=" * 70)
print("ГОТОВО!")
print("=" * 70)
print(f"Все файлы в папке: {OUTPUT_DIR}/")
print()
print("Созданные файлы:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    print(f"  {f}")

ТАБЛИЦА РАБОТ
Работа  Длит  Ресурсы  Резерв  Начало  Конец  Ресурсо-дни
   1,2    10        4       0       0     10           40
   1,4     1        2      21       0      1            2
   1,5     5        2      15       0      5           10
   2,3     9        3       0      10     19           27
   2,5     8        2       2      10     18           16
   2,6    10        5       7      10     20           50
   3,4     3        1       0      19     22            3
   3,6     4        2       4      19     23            8
   4,6     5        3       0      22     27           15
   4,7     4        5       9      22     26           20
   5,6     7        2       2      18     25           14
   5,7     3        2      14      18     21            6
   6,7     8        2       0      27     35           16

Всего ресурсо-дней: 227
Критический путь: 35 дней
Среднее: 227 / 35 = 6.49

РАСПРЕДЕЛЕНИЕ РЕСУРСОВ (без сдвига)
       1   2   3   4   5   6   7   8   9   10  11  12  13  14

In [8]:
"""
Лабораторная работа 1. Сетевое планирование.
Вариант 3.

Всё как в примере: имитационная модель распределения ресурсов,
работы ждут при нехватке ресурсов, проект удлиняется.
"""

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import numpy as np
import os
from copy import deepcopy

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False

OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# 1. ИСХОДНЫЕ ДАННЫЕ
# ============================================================

# (Работа, Длит_норм, Ресурсы, Резерв, Длит_макс, Затраты_норм, Затраты_макс)
raboty = [
    ("1,2", 10, 4, 0, 8, 60, 66),
    ("1,4", 1, 2, 21, 0.5, 85, 90),
    ("1,5", 5, 2, 15, 3.5, 100, 120),
    ("2,3", 9, 3, 0, 6, 90, 105),
    ("2,5", 8, 2, 2, 7, 55, 70),
    ("2,6", 10, 5, 7, 9, 35, 50),
    ("3,4", 3, 1, 0, 2, 40, 60),
    ("3,6", 4, 2, 4, 2.5, 48, 58),
    ("4,6", 5, 3, 0, 3, 95, 107),
    ("4,7", 4, 5, 9, 2, 38, 50),
    ("5,6", 7, 2, 2, 6, 60, 70),
    ("5,7", 3, 2, 14, 1.5, 24, 30),
    ("6,7", 8, 2, 0, 5.5, 52, 62),
]

df = pd.DataFrame(raboty, columns=[
    "Работа", "Длит", "Ресурсы", "Резерв", "Длит_макс", "З_норм", "З_макс"
])

# Ранние сроки начала (из расчёта сетевого графика)
rannee_nachalo = {
    "1,2": 0, "1,4": 0, "1,5": 0,
    "2,3": 10, "2,5": 10, "2,6": 10,
    "3,4": 19, "3,6": 19,
    "4,6": 22, "4,7": 22,
    "5,6": 18, "5,7": 18,
    "6,7": 27,
}

df["Начало"] = df["Работа"].map(rannee_nachalo)
df["Конец"] = df["Начало"] + df["Длит"]
df["Ресурсо-дни"] = df["Длит"] * df["Ресурсы"]

total_res_days = df["Ресурсо-дни"].sum()
T_kr = int(df["Конец"].max())

print("=" * 70)
print("ТАБЛИЦА РАБОТ")
print("=" * 70)
print(df.to_string(index=False))
print()
print(f"Всего ресурсо-дней: {total_res_days}")
print(f"Критический путь: {T_kr} дней")
print(f"Среднее число ресурсов: {total_res_days} / {T_kr} = "
      f"{total_res_days / T_kr:.2f} ≈ {round(total_res_days / T_kr)}")
print()

# ============================================================
# 2. ДИАГРАММА ГАНТА
# ============================================================

def build_gantt(df, filename):
    fig, ax = plt.subplots(figsize=(14, 8))
    for i, row in df.iterrows():
        if row["Резерв"] > 0:
            ax.barh(i, row["Резерв"], left=row["Конец"],
                    color="white", edgecolor="black",
                    linewidth=1.2, linestyle="--", height=1.0)
        ax.barh(i, row["Длит"], left=row["Начало"],
                color="lightgray", edgecolor="black",
                linewidth=1.2, height=1.0, hatch="///")
    ax.set_yticks(range(len(df)))
    ax.set_yticklabels(df["Работа"])
    ax.set_xlabel("Дни", fontsize=12)
    ax.set_ylabel("Работы", fontsize=12)
    ax.set_title("Диаграмма Ганта", fontsize=14)
    ax.set_xlim(0, T_kr + 2)
    ax.set_xticks(range(0, T_kr + 2, 2))
    ax.grid(axis="x", linestyle=":", alpha=0.4)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    legend_elements = [
        mpatches.Patch(facecolor="lightgray", edgecolor="black",
                       hatch="///", label="Работа"),
        mpatches.Patch(facecolor="white", edgecolor="black",
                       linestyle="--", label="Резерв"),
    ]
    ax.legend(handles=legend_elements, loc="lower right")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, filename), dpi=200)
    plt.close()
    print(f"  ✅ {filename}")

build_gantt(df, "gantt.png")

# ============================================================
# 3. ИМИТАЦИОННАЯ МОДЕЛЬ РАСПРЕДЕЛЕНИЯ РЕСУРСОВ
# ============================================================

def simulate_resources(df, limit, max_sim_days=200):
    """
    Имитационная модель: работы ждут, если ресурсов не хватает.
    Возвращает:
    - schedule: DataFrame с полями (Работа, Начало, Конец, Ресурсы)
    - table: таблица распределения по дням
    - total_days: срок проекта
    """
    schedule = df[["Работа", "Длит", "Ресурсы", "Резерв"]].copy()
    schedule["Текущее_начало"] = None
    schedule["Текущий_конец"] = None
    schedule["Статус"] = "ожидает"

    predecessors = {
        "1,2": [],
        "1,4": [],
        "1,5": [],
        "2,3": ["1,2"],
        "2,5": ["1,2"],
        "2,6": ["1,2"],
        "3,4": ["2,3"],
        "3,6": ["2,3"],
        "4,6": ["3,4"],
        "4,7": ["3,4"],
        "5,6": ["2,5", "1,5"],
        "5,7": ["2,5", "1,5"],
        "6,7": ["2,6", "3,6", "4,6", "5,6"],
    }

    idx_map = {row["Работа"]: i for i, row in schedule.iterrows()}

    current_day = 0
    while current_day < max_sim_days:
        current_day += 1

        for i, row in schedule.iterrows():
            if row["Статус"] == "идёт" and current_day > row["Текущий_конец"]:
                schedule.at[i, "Статус"] = "готово"

        used = sum(row["Ресурсы"] for _, row in schedule.iterrows()
                   if row["Статус"] == "идёт")
        free = limit - used

        ready = []
        for i, row in schedule.iterrows():
            if row["Статус"] != "ожидает":
                continue
            preds = predecessors[row["Работа"]]
            preds_done = all(
                schedule.loc[idx_map[p], "Статус"] == "готово"
                for p in preds
            )
            if preds_done:
                ready.append(i)

        ready.sort(key=lambda i: (
            schedule.loc[i, "Резерв"],
            -schedule.loc[i, "Ресурсы"],
        ))

        for i in ready:
            row = schedule.loc[i]
            if row["Ресурсы"] <= free:
                schedule.at[i, "Текущее_начало"] = current_day
                schedule.at[i, "Текущий_конец"] = current_day + row["Длит"] - 1
                schedule.at[i, "Статус"] = "идёт"
                free -= row["Ресурсы"]

        if all(row["Статус"] == "готово" for _, row in schedule.iterrows()):
            break

    total_days = int(schedule["Текущий_конец"].max())

    days = list(range(1, total_days + 1))
    table_data = {}
    for i, row in schedule.iterrows():
        row_data = []
        for d in days:
            if (row["Текущее_начало"] is not None
                    and row["Текущее_начало"] <= d <= row["Текущий_конец"]):
                row_data.append(row["Ресурсы"])
            else:
                row_data.append(0)
        table_data[row["Работа"]] = row_data
    table_data["СУММА"] = [
        sum(table_data[r][i] for r in table_data if r != "СУММА")
        for i in range(len(days))
    ]
    table = pd.DataFrame(table_data, index=days).T

    return schedule, table, total_days

# ============================================================
# 4. ГИСТОГРАММА
# ============================================================

def build_resource_hist(table, filename, limit, total_days):
    days = table.columns.tolist()
    values = table.loc["СУММА"].tolist()
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.bar(days, values, color="lightgray", edgecolor="black", width=0.9)
    ax.axhline(limit, color="red", linestyle="--", linewidth=2,
               label=f"Лимит = {limit}")
    sred = sum(values) / len(values)
    ax.axhline(sred, color="blue", linestyle=":", linewidth=1.5,
               label=f"Среднее = {sred:.2f}")
    ax.set_xlabel("Дни", fontsize=12)
    ax.set_ylabel("Ресурсы, чел.", fontsize=12)
    ax.set_title(f"Распределение ресурсов (лимит = {limit}, "
                 f"срок = {total_days} дней)", fontsize=14)
    ax.set_xticks(days)
    ax.set_ylim(0, max(values) + 2)
    ax.grid(axis="y", linestyle=":", alpha=0.4)
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, filename), dpi=200)
    plt.close()
    print(f"  ✅ {filename}")

# ============================================================
# 5. ПРОГОНЯЕМ ДЛЯ РАЗНЫХ ЛИМИТОВ
# ============================================================

limits = [6, 7, 8, 9, 10]
results = {}

print()
print("=" * 70)
print("ИМИТАЦИОННАЯ МОДЕЛЬ ДЛЯ РАЗНЫХ ЛИМИТОВ")
print("=" * 70)

for lim in limits:
    schedule, table, total_days = simulate_resources(df, lim)
    peak = table.loc["СУММА"].max()
    results[lim] = {
        "schedule": schedule,
        "table": table,
        "total_days": total_days,
        "peak": peak,
    }
    print(f"Лимит = {lim}: срок = {total_days} дней, пик = {peak}")

    table.to_excel(os.path.join(OUTPUT_DIR, f"resursy_{lim}.xlsx"))
    schedule.to_excel(os.path.join(OUTPUT_DIR, f"raboty_{lim}.xlsx"), index=False)

    build_resource_hist(table, f"resursy_{lim}.png", lim, total_days)

# ============================================================
# 6. ГРАФИКИ КРЕДИТА И ПРИБЫЛИ
# ============================================================

# --- Нормальный режим (срок = 35 дней) ---
stoimost_rabot_norm = df["З_норм"].sum()  # 782
ostatok_norm = 2000 - stoimost_rabot_norm  # 1218
pribyl_v_den = 20 * 2  # 40 д.е./день

x_C, y_C = 35, ostatok_norm  # (35; 1218)
x_D, y_D = 45, ostatok_norm + pribyl_v_den * 10  # (45; 1618)

k = (y_D - y_C) / (x_D - x_C)
b = y_C - k * x_C  # b = 1218 - 40*35 = -182

x_cross = (b - 2000) / (10/9 - k)
y_cross = (10/9) * x_cross + 2000

fig, ax = plt.subplots(figsize=(14, 8))
x = np.linspace(0, 200, 500)
kredit = (10/9) * x + 2000
pribyl = k * x + b
pribyl[pribyl < -1000] = np.nan

ax.plot(x, kredit, "b-", linewidth=2.5, label="Кредит")
ax.plot(x, pribyl, "g-", linewidth=2.5, label="Прибыль (норм. режим)")
ax.plot(x_cross, y_cross, "ko", markersize=10)
ax.annotate(f"({x_cross:.1f}; {y_cross:.1f})",
            xy=(x_cross, y_cross),
            xytext=(x_cross - 100, y_cross + 300),
            arrowprops=dict(arrowstyle="->", color="black", lw=1.5),
            fontsize=12)

ax.set_xlabel("Дни", fontsize=13)
ax.set_ylabel("Стоимость, д.е.", fontsize=13)
ax.set_title("Кредит и прибыль (нормальный режим)", fontsize=14)
ax.grid(True, linestyle=":", alpha=0.5)
ax.legend(loc="upper left", fontsize=12)
ax.axhline(0, color="black", linewidth=0.8)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "credit_norm.png"), dpi=200)
plt.close()
print("  ✅ credit_norm.png")

# --- Максимальный режим ---
sokrash = {"1,2": 2, "2,3": 3, "3,4": 1, "4,6": 2, "6,7": 2.5}
srok_max = 35 - sum(sokrash.values())  # 24.5
dop_zatraty = sum(
    df.loc[df["Работа"] == r, "З_макс"].values[0]
    - df.loc[df["Работа"] == r, "З_норм"].values[0]
    for r in sokrash
)
stoimost_max = stoimost_rabot_norm + dop_zatraty  # 845
ostatok_max = 2000 - stoimost_max  # 1155

x_C2, y_C2 = srok_max, ostatok_max  # (24.5; 1155)
x_D2, y_D2 = srok_max + 10, ostatok_max + pribyl_v_den * 10  # (34.5; 1555)
k2 = (y_D2 - y_C2) / (x_D2 - x_C2)
b2 = y_C2 - k2 * x_C2  # b2 = 1155 - 40*24.5 = 175

x_cross2 = (b2 - 2000) / (10/9 - k2)
y_cross2 = (10/9) * x_cross2 + 2000

fig, ax = plt.subplots(figsize=(14, 8))
x = np.linspace(0, 200, 500)
kredit = (10/9) * x + 2000
pribyl = k2 * x + b2
pribyl[pribyl < -1000] = np.nan

ax.plot(x, kredit, "b-", linewidth=2.5, label="Кредит")
ax.plot(x, pribyl, "r-", linewidth=2.5, label="Прибыль (макс. режим)")
ax.plot(x_cross2, y_cross2, "ko", markersize=10)
ax.annotate(f"({x_cross2:.1f}; {y_cross2:.1f})",
            xy=(x_cross2, y_cross2),
            xytext=(x_cross2 - 100, y_cross2 + 300),
            arrowprops=dict(arrowstyle="->", color="black", lw=1.5),
            fontsize=12)

ax.set_xlabel("Дни", fontsize=13)
ax.set_ylabel("Стоимость, д.е.", fontsize=13)
ax.set_title("Кредит и прибыль (максимальный режим)", fontsize=14)
ax.grid(True, linestyle=":", alpha=0.5)
ax.legend(loc="upper left", fontsize=12)
ax.axhline(0, color="black", linewidth=0.8)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "credit_max.png"), dpi=200)
plt.close()
print("  ✅ credit_max.png")

# ============================================================
# 7. ИТОГОВЫЙ EXCEL
# ============================================================

with pd.ExcelWriter(os.path.join(OUTPUT_DIR, "otchet.xlsx")) as writer:
    df.to_excel(writer, sheet_name="Работы", index=False)
    for lim in limits:
        results[lim]["table"].to_excel(writer, sheet_name=f"Ресурсы_{lim}")
        results[lim]["schedule"].to_excel(
            writer, sheet_name=f"Работы_{lim}", index=False
        )

# ============================================================
# 8. ИТОГ
# ============================================================

print()
print("=" * 70)
print("ИТОГИ")
print("=" * 70)
print(f"Всего ресурсо-дней: {total_res_days}")
print(f"Критический путь: {T_kr} дней")
print(f"Среднее число ресурсов: {total_res_days / T_kr:.2f}")
print()
print("Сроки для разных лимитов:")
for lim in limits:
    print(f"  {lim} чел. → {results[lim]['total_days']} дней "
          f"(пик {results[lim]['peak']})")
print()
print("Нормальный режим:")
print(f"  Срок: {T_kr} дней")
print(f"  Стоимость работ: {stoimost_rabot_norm:.1f}")
print(f"  Остаток: {ostatok_norm:.1f}")
print(f"  Точка возврата кредита: {x_cross:.1f} дней, {y_cross:.1f} д.е.")
print()
print("Максимальный режим:")
print(f"  Срок: {srok_max:.1f} дней")
print(f"  Стоимость работ: {stoimost_max:.1f}")
print(f"  Остаток: {ostatok_max:.1f}")
print(f"  Точка возврата кредита: {x_cross2:.1f} дней, {y_cross2:.1f} д.е.")
print()
print("=" * 70)
print("ГОТОВО!")
print("=" * 70)
print(f"Все файлы в папке: {OUTPUT_DIR}/")
for f in sorted(os.listdir(OUTPUT_DIR)):
    print(f"  {f}")

ТАБЛИЦА РАБОТ
Работа  Длит  Ресурсы  Резерв  Длит_макс  З_норм  З_макс  Начало  Конец  Ресурсо-дни
   1,2    10        4       0        8.0      60      66       0     10           40
   1,4     1        2      21        0.5      85      90       0      1            2
   1,5     5        2      15        3.5     100     120       0      5           10
   2,3     9        3       0        6.0      90     105      10     19           27
   2,5     8        2       2        7.0      55      70      10     18           16
   2,6    10        5       7        9.0      35      50      10     20           50
   3,4     3        1       0        2.0      40      60      19     22            3
   3,6     4        2       4        2.5      48      58      19     23            8
   4,6     5        3       0        3.0      95     107      22     27           15
   4,7     4        5       9        2.0      38      50      22     26           20
   5,6     7        2       2        6.0      60   

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import numpy as np

# ============================================================
# 1. ИСХОДНЫЕ ДАННЫЕ
# ============================================================

# (название, начало, длительность, ресурсы, резерв)
raboty = [
    ("1,2", 0, 10, 4, 0),
    ("1,4", 0, 1, 2, 21),
    ("1,5", 0, 5, 2, 15),
    ("2,3", 10, 9, 3, 0),
    ("2,5", 10, 8, 2, 2),
    ("2,6", 10, 10, 5, 7),
    ("3,4", 19, 3, 1, 0),
    ("3,6", 19, 4, 2, 4),
    ("4,6", 22, 5, 3, 0),
    ("4,7", 22, 4, 5, 9),
    ("5,6", 18, 7, 2, 2),
    ("5,7", 18, 3, 2, 14),
    ("6,7", 27, 8, 2, 0),
]

df = pd.DataFrame(raboty, columns=["Работа", "Начало", "Длительность", "Ресурсы", "Резерв"])
df["Конец"] = df["Начало"] + df["Длительность"]

# ВАЖНО: порядок снизу вверх — 1,2 внизу, 6,7 наверху
# В matplotlib ось Y идёт снизу вверх, поэтому оставляем как есть
# (первая работа — внизу)

print("Таблица работ:")
print(df)
print()

# ============================================================
# 2. ДИАГРАММА ГАНТА (точно как в PDF)
# ============================================================

fig, ax = plt.subplots(figsize=(14, 8))

for i, row in df.iterrows():
    # --- 1. Пунктирная часть (резерв) ---
    if row["Резерв"] > 0:
        # Резерв рисуется ОТ КОНЦА работы ДО КОНЦА + РЕЗЕРВ
        ax.barh(
            y=i,
            width=row["Резерв"],
            left=row["Конец"],
            color="white",           # белый фон
            edgecolor="black",       # чёрная рамка
            linewidth=1.5,
            linestyle="--",          # пунктир
            height=1,
        )
    
    # --- 2. Сплошная часть (работа) ---
    # ОДИНАКОВАЯ для всех: серая заливка + штриховка
    ax.barh(
        y=i,
        width=row["Длительность"],
        left=row["Начало"],
        color="lightgray",           # серый
        edgecolor="black",           # чёрная рамка
        linewidth=1.5,
        height=1,
        hatch="///",                 # одинаковая штриховка для ВСЕХ
    )

# ============================================================
# 3. НАСТРОЙКИ ОСЕЙ
# ============================================================

# Подписи работ по оси Y
ax.set_yticks(range(len(df)))
ax.set_yticklabels(df["Работа"], fontsize=11)

# Оси
ax.set_xlabel("Дни", fontsize=13)
ax.set_ylabel("Работы", fontsize=13)
ax.set_title("Диаграмма Ганта", fontsize=15)

# Пределы оси X
ax.set_xlim(0, 36)

# Деления по X каждые 2 дня
ax.set_xticks(range(0, 37, 2))
ax.tick_params(axis="x", labelsize=10)

# Сетка по X (лёгкая, чтобы читалось)
ax.grid(axis="x", linestyle=":", alpha=0.4)

# Убираем лишние рамки
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# ============================================================
# 4. ЛЕГЕНДА
# ============================================================

legend_elements = [
    mpatches.Patch(
        facecolor="lightgray",
        edgecolor="black",
        hatch="///",
        label="Работа",
    ),
    mpatches.Patch(
        facecolor="white",
        edgecolor="black",
        linestyle="--",
        label="Резерв времени",
    ),
]
ax.legend(handles=legend_elements, loc="lower right", fontsize=11)

plt.tight_layout()
plt.savefig("gantt.png", dpi=150)
plt.show()

In [ ]:
"""
Лабораторная работа 1. Сетевое планирование.
Вариант 3.
"""

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import numpy as np

# ============================================================
# 1. ИСХОДНЫЕ ДАННЫЕ
# ============================================================

# (Работа, Начало, Длительность, Ресурсы, Резерв)
raboty = [
    ("1,2", 0, 10, 4, 0),
    ("1,4", 0, 1, 2, 21),
    ("1,5", 0, 5, 2, 15),
    ("2,3", 10, 9, 3, 0),
    ("2,5", 10, 8, 2, 2),
    ("2,6", 10, 10, 5, 7),
    ("3,4", 19, 3, 1, 0),
    ("3,6", 19, 4, 2, 4),
    ("4,6", 22, 5, 3, 0),
    ("4,7", 22, 4, 5, 9),
    ("5,6", 18, 7, 2, 2),
    ("5,7", 18, 3, 2, 14),
    ("6,7", 27, 8, 2, 0),
]

df = pd.DataFrame(raboty, columns=["Работа", "Начало", "Длительность", "Ресурсы", "Резерв"])
df["Конец"] = df["Начало"] + df["Длительность"]

print("=" * 60)
print("ТАБЛИЦА РАБОТ")
print("=" * 60)
print(df.to_string(index=False))
print()

# ============================================================
# 2. ДИАГРАММА ГАНТА
# ============================================================

fig1, ax1 = plt.subplots(figsize=(14, 8))

for i, row in df.iterrows():
    # Резерв (пунктирная рамка)
    if row["Резерв"] > 0:
        ax1.barh(
            y=i,
            width=row["Резерв"],
            left=row["Конец"],
            color="white",
            edgecolor="black",
            linewidth=1.2,
            linestyle="--",
            height=1.0,
        )
    # Работа (серая + штриховка)
    ax1.barh(
        y=i,
        width=row["Длительность"],
        left=row["Начало"],
        color="lightgray",
        edgecolor="black",
        linewidth=1.2,
        height=1.0,
        hatch="///",
    )

ax1.set_yticks(range(len(df)))
ax1.set_yticklabels(df["Работа"], fontsize=11)
ax1.set_xlabel("Дни", fontsize=13)
ax1.set_ylabel("Работы", fontsize=13)
ax1.set_title("Диаграмма Ганта", fontsize=15)
ax1.set_xlim(0, 36)
ax1.set_xticks(range(0, 37, 2))
ax1.grid(axis="x", linestyle=":", alpha=0.4)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

legend_elements = [
    mpatches.Patch(facecolor="lightgray", edgecolor="black", hatch="///", label="Работа"),
    mpatches.Patch(facecolor="white", edgecolor="black", linestyle="--", label="Резерв времени"),
]
ax1.legend(handles=legend_elements, loc="lower right", fontsize=11)

plt.tight_layout()
plt.savefig("gantt.png", dpi=200)
plt.show()
print("Сохранено: gantt.png")

# ============================================================
# 3. РАСПРЕДЕЛЕНИЕ РЕСУРСОВ ПО ДНЯМ
# ============================================================

max_day = int(df["Конец"].max())  # 35
days = np.arange(1, max_day + 1)

# Считаем сумму ресурсов в каждый день
resursy_po_dnyam = []
for day in days:
    total = 0
    for _, row in df.iterrows():
        if row["Начало"] < day <= row["Конец"]:
            total += row["Ресурсы"]
    resursy_po_dnyam.append(total)

# Таблица распределения
df_res = pd.DataFrame({
    "День": days,
    "Ресурсы": resursy_po_dnyam,
})

print()
print("=" * 60)
print("РАСПРЕДЕЛЕНИЕ РЕСУРСОВ ПО ДНЯМ")
print("=" * 60)
print(df_res.to_string(index=False))
print()
print(f"Максимум ресурсов: {max(resursy_po_dnyam)}")
print(f"Среднее: {sum(resursy_po_dnyam) / len(resursy_po_dnyam):.2f}")

# Сохраняем в Excel
df_res.to_excel("resursy_po_dnyam.xlsx", index=False)
print("Сохранено: resursy_po_dnyam.xlsx")

# ============================================================
# 4. ГИСТОГРАММА РЕСУРСОВ
# ============================================================

fig2, ax2 = plt.subplots(figsize=(14, 6))

ax2.bar(
    days,
    resursy_po_dnyam,
    color="seagreen",
    edgecolor="black",
    width=0.9,
)

srednee = sum(resursy_po_dnyam) / len(resursy_po_dnyam)
ax2.axhline(srednee, color="red", linestyle="--", linewidth=1.5,
            label=f"Среднее = {srednee:.2f}")

ax2.set_xlabel("Дни", fontsize=13)
ax2.set_ylabel("Ресурсы, чел.", fontsize=13)
ax2.set_title("Диаграмма распределения ресурсов", fontsize=15)
ax2.set_xticks(range(1, max_day + 1, 2))
ax2.set_ylim(0, max(resursy_po_dnyam) + 2)
ax2.grid(axis="y", linestyle=":", alpha=0.4)
ax2.legend(fontsize=11)

plt.tight_layout()
plt.savefig("resursy.png", dpi=200)
plt.show()
print("Сохранено: resursy.png")

# ============================================================
# 5. ГРАФИК КРЕДИТА И ПРИБЫЛИ
# ============================================================

fig3, ax3 = plt.subplots(figsize=(14, 8))

x = np.linspace(0, 800, 500)

# Кредит: y = (10/9)x + 2000
kredit = (10/9) * x + 2000

# Прибыль (0,5 д.е./т): y = 10x - 4655
pribyl_05 = 10 * x - 4655
pribyl_05[pribyl_05 < 0] = np.nan

# Прибыль (2 д.е./т): y = 40x - 6205
pribyl_2 = 40 * x - 6205
pribyl_2[pribyl_2 < 0] = np.nan

ax3.plot(x, kredit, "b-", linewidth=2.5, label="Кредит")
ax3.plot(x, pribyl_05, "r-", linewidth=2.5, label="Прибыль (0,5 д.е./т)")
ax3.plot(x, pribyl_2, "g--", linewidth=2.5, label="Прибыль (2 д.е./т)")

# Точка пересечения для 0,5: x ≈ 748,7; y ≈ 2832
ax3.plot(748.7, 2832, "ko", markersize=10)
ax3.annotate(
    "(748,7; 2832)",
    xy=(748.7, 2832),
    xytext=(550, 2600),
    arrowprops=dict(arrowstyle="->", color="black", lw=1.5),
    fontsize=12,
)

ax3.set_xlabel("Дни", fontsize=13)
ax3.set_ylabel("Стоимость, д.е.", fontsize=13)
ax3.set_title("Зависимость кредита и прибыли от времени", fontsize=15)
ax3.set_xlim(0, 800)
ax3.set_ylim(-2000, 3500)
ax3.grid(True, linestyle=":", alpha=0.5)
ax3.legend(loc="upper left", fontsize=12)
ax3.axhline(0, color="black", linewidth=0.8)

plt.tight_layout()
plt.savefig("credit.png", dpi=200)
plt.show()
print("Сохранено: credit.png")

# ============================================================
# 6. СОХРАНЕНИЕ ТАБЛИЦ В EXCEL
# ============================================================

with pd.ExcelWriter("otchet.xlsx") as writer:
    df.to_excel(writer, sheet_name="Работы", index=False)
    df_res.to_excel(writer, sheet_name="Ресурсы по дням", index=False)

print("Сохранено: otchet.xlsx")
print()
print("=" * 60)
print("ГОТОВО!")
print("=" * 60)
print("Файлы:")
print("  gantt.png         — диаграмма Ганта")
print("  resursy.png       — гистограмма ресурсов")
print("  credit.png        — график кредита и прибыли")
print("  otchet.xlsx       — таблицы")